In [19]:
import pandas as pd
import numpy as np
from pathlib import Path

In [51]:
SPECTROMETER_FREQUENCY_MHZ = 600.0
CSV_FILE_PATHS = sorted(Path("data/mnova_data").glob("*.csv"))
OUTPUT_DIR = Path("data/processed_spectra")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [58]:
def parse_file(file_path):
    lines = Path(file_path).read_text().splitlines()[1:] # skip header

    peaks_data = []
    for line in lines:
        line_parts = line.split()
        # line_data = {
        #     "name": line_parts[1],
        #     "from_ppm": float(line_parts[2]),
        #     "to_ppm": float(line_parts[3]),
        #     "error": float(line_parts[4]),
        # }
        peaks_data.extend([
            {
                "multiplet_name": line_parts[1],
                "peak_in_multiplet": int(line_parts[i]),
                "position_ppm": float(line_parts[i + 1]),
                "height": float(line_parts[i + 2]),
                "width_hz": float(line_parts[i + 3]),
                "L/G": float(line_parts[i + 4]),
                "area": float(line_parts[i + 5]),
                }
            for i in range(5, len(line_parts), 6)
        ])

    # for peak_data in peaks_data:
    #     l_g = peak_data.pop("L/G")
    #     # print(peak_data["name"], l_g)
    #     peak_data["gaussian_fraction"] = np.clip(1 / (1 + np.clip(l_g, 0, None)), 0.0, 1.0)
        
    return peaks_data



In [59]:
for csv_file_path in CSV_FILE_PATHS:
    peaks_data = parse_file(csv_file_path)
    output_file_path = OUTPUT_DIR / (Path(csv_file_path).stem + ".csv")
    peaks_data = pd.DataFrame(peaks_data)
    peaks_data["position_hz"] = peaks_data["position_ppm"] * SPECTROMETER_FREQUENCY_MHZ
    peaks_data["gaussian_fraction"] = np.clip(1 / (1 + np.clip(peaks_data["L/G"], 0, None)), 0.0, 1.0)
    peaks_data = peaks_data.drop(columns=["L/G"])
    peaks_data.to_csv(output_file_path, index=False)